# How to run L-C2ST

L-C2ST tests whether your posterior is correct **for one observation** `x_o`. Expected coverage and SBC instead check the posterior on average over many observations. L-C2ST trains a classifier on new simulations and needs no samples from the true posterior.

For how the test works and how to read its diagnostics in depth, see the [L-C2ST tutorial](https://sbi.readthedocs.io/en/latest/advanced_tutorials/13_diagnostics_lc2st.html).

## Run the test

```python
from sbi.diagnostics import LC2ST

# `prior`, `simulator` and a trained amortized `posterior` come from your workflow.

# 1. Simulate new calibration data, not the data the posterior was trained on,
#    and draw one posterior sample for each simulation.
num_cal = 10_000
theta_cal = prior.sample((num_cal,))
x_cal = simulator(theta_cal)
post_samples_cal = posterior.sample_batched((1,), x=x_cal)[0]

# 2. Train the classifiers once.
lc2st = LC2ST(prior_samples=theta_cal, xs=x_cal, posterior_samples=post_samples_cal)
lc2st.train_on_observed_data()
lc2st.train_under_null_hypothesis()

# 3. Test the posterior at an observation `x_o` (a tensor).
post_samples_o = posterior.sample((10_000,), x=x_o)
p_value = lc2st.p_value(post_samples_o, x_o)
reject = lc2st.reject_test(post_samples_o, x_o, alpha=0.05)
```

If `reject` is `True`, the test finds that the posterior at `x_o` differs from the true posterior. The test can also reject when the posterior is wrong only at observations near `x_o`, because the classifier learns from all of them; the tutorial explains why.

- **Calibration data:** use thousands of simulations that the posterior has not seen. The test is only as sensitive as the classifier trained on them.
- **Cost:** `train_under_null_hypothesis` trains 100 classifiers on shuffled data (`num_trials_null`). This is the slow step. Pass `device="cuda"` to `LC2ST` to train the default MLP classifiers on a GPU.
- **Batched sampling:** step 1 needs a posterior with `sample_batched`. If yours has none, draw one sample per calibration simulation in a loop.
- **Many observations:** the trained `lc2st` works for any observation, so only step 3 runs again for a new `x_o`.
- **Observations with more than one dimension**, such as images or time series: the classifier takes parameters and observations as flat vectors. Pass flattened observations to `LC2ST` and its methods, and the original ones to the posterior.

## See how the posterior is off

If the test rejects, a P-P plot of the classifier's predicted probabilities shows whether the posterior is over- or under-confident at `x_o`:

```python
from sbi.analysis.plot import pp_plot_lc2st

probs_data = lc2st.get_scores(
    theta_o=post_samples_o, x_o=x_o, trained_clfs=lc2st.trained_clfs
).probabilities
probs_null = lc2st.get_statistics_under_null_hypothesis(
    theta_o=post_samples_o, x_o=x_o
).probabilities

pp_plot_lc2st(
    probs=[probs_data],
    probs_null=probs_null,
    conf_alpha=0.05,
    labels=["Classifier probabilities \n on observed data"],
    colors=["red"],
)
```

<img src="data/L_C2ST_pp_plot.png" width="500">

The gray band shows where the red curve lies when the posterior is correct. If the red curve is below the band, the classifier gives high probabilities to the posterior samples: the posterior puts too much mass where it samples (over-confident in that sense). If the curve is above the band, it puts too little mass there (under-confident). Either can come from a posterior that is too narrow, too wide or shifted. The plot is a visual diagnostic; the test decision comes from `reject`. To see in which region of parameter space the posterior is off, plot the probabilities over the posterior samples, as the tutorial shows.

## Options

- `num_folds` and `num_ensemble`: cross-validate, or average several classifiers. Both make the test less noisy and cost more training.
- `classifier`: `"mlp"` (default), `"random_forest"`, or any scikit-learn classifier class with a `predict_proba` method, with its arguments in `classifier_kwargs`.
- `LC2ST_NF`: a variant for normalizing-flow posteriors. It runs the test in the base space of the flow, where the null distribution is known, so its null classifiers do not depend on the posterior: you can reuse them to test other flows on the same calibration simulations, with the same base distribution and settings, and the default `z_score=False`. It tests the flow itself, so pass it flow samples drawn with `reject_outside_prior=False`; to test the posterior after `sbi` rejects samples outside a bounded prior, use `LC2ST`. The tutorial shows how to set it up.

## Citation

```text
@article{linhart2023c2st,
  title={L-c2st: Local diagnostics for posterior approximations in simulation-based inference},
  author={Linhart, Julia and Gramfort, Alexandre and Rodrigues, Pedro},
  journal={Advances in Neural Information Processing Systems},
  volume={36},
  pages={56384--56410},
  year={2023}
}
```